In [24]:
import requests
import pandas as pd
from io import StringIO
import pickle
import gzip

In [23]:
class PSKReporter():
    url = "https://retrieve.pskreporter.info/query"

    def __init__(self, callsign):
        self.senderCallsign = callsign

    def get_report(self, time=30):

        params = {
            "senderCallsign" : self.senderCallsign,
            "flowStartSeconds": time * (-60)
        }

        # Sending a request to PSK Reporter.
        try:
            r = requests.get(self.url, params=params)

            report = {}

            # If the XML is HTML-escaped (e.g., &lt; and &gt;), decode it first
            xml_string = r.text.replace("&lt;", "<").replace("&gt;", ">")

            # Reports that contains the given callsign.
            try:
                df_reception_report = pd.read_xml(StringIO(xml_string), xpath=".//receptionReport")
            except ValueError as ve:
                print("No reception reports...")
                df_reception_report = pd.DataFrame()

            report['reception_reports'] = df_reception_report

            # Callsigns that were recently reported as active.
            df_active_cs = pd.read_xml(StringIO(xml_string), xpath=".//activeCallsign")
            report['active_cs'] = df_active_cs

            # This are the stations that are currently active.
            df_active_receivers = pd.read_xml(StringIO(xml_string), xpath=".//activeReceiver")
            df_active_receivers_reduced = df_active_receivers[['callsign', 'locator', 'frequency', 'mode']]
            df_active_receivers_reduced = df_active_receivers_reduced[df_active_receivers_reduced['mode'] == "FT8"]
            report['active_receivers'] = df_active_receivers_reduced

            # Contains the senderCallsign and the most recent unix epoch of when a transmission from senderCallsign was reported.
            #df_sender_search = pd.read_xml(StringIO(xml_string), xpath=".//senderSearch")

            # Unique identifier for the PSK Reporter request, not really useful for the dataset.
            #df_last_sequence_number = pd.read_xml(StringIO(xml_string), xpath=".//lastSequenceNumber")

            # Unix epoch of the last report contained in this response.
            df_max_flow_start_seconds = pd.read_xml(StringIO(xml_string), xpath=".//maxFlowStartSeconds")
            report['last_report_time'] = df_max_flow_start_seconds['value'][0]

            print(report)

        except ValueError as ve:
            print(f"Unable to query data for this sample! {ve}")
            report = -1

        return report


In [9]:
psk = PSKReporter('2E1LSI')

In [10]:
report = psk.get_report(60)

No reception reports...
{'reception_reports': Empty DataFrame
Columns: []
Index: [], 'active_cs':    callsign  reports             DXCC DXCCcode  frequency
0    KF8DPO        1    United States        K   14076908
1     K5BYN        1    United States        K   14075745
2      WD4U        1    United States        K   14075941
3     LA9BM        1           Norway       LA   50313342
4     PY2FZ        1           Brazil       PY   28075337
5     KE2UK        1    United States        K   14080860
6     R5VDX        1  European Russia       UA   14081083
7     R0ACR        1   Asiatic Russia      UA9   14080715
8      KI0E        1    United States        K   14080730
9     R9HEB        2   Asiatic Russia      UA9   21075449
10   KO8SCA        3    United States        K   14076909
11    F1PGQ        1           France        F   18100634
12   WB0WAO        2    United States        K   14075045
13     K1VP        1    United States        K   14075140
14     R6OI        1  European R

In [11]:
report['active_receivers']

,callsign,locator,frequency,mode
0,RN8C,MO06lq,14075373.0,FT8
1,SM6ZDM,JO67du,21076481.0,FT8
2,SM6YEC,JO67AJ,14075480.0,FT8
3,MI0UFT,IO64IO,14075360.0,FT8
4,KY0R,DM78PF,14075079.0,FT8
...,...,...,...,...
7135,KF0UQR,EM18fe88,50313613.0,FT8
7136,OK1MXD,JN89AX,50313797.0,FT8
7137,PA0RXO,JO20XV,50313624.0,FT8
7138,RA3CQ,KO85sp,50313820.0,FT8


In [12]:
report['active_receivers'].memory_usage(deep=True).sum()

np.int64(1216149)

In [13]:
size_bytes = report['active_receivers'].memory_usage(deep=True).sum()
size_mb = size_bytes / (1024 ** 2)
print(f"DataFrame size: {size_bytes} bytes ({size_mb:.2f} MB)")

DataFrame size: 1216149 bytes (1.16 MB)


In [16]:
final_report = report['active_receivers'][report['active_receivers']['mode'] == "FT8"]

In [17]:
final_report

,callsign,locator,frequency,mode
0,RN8C,MO06lq,14075373.0,FT8
1,SM6ZDM,JO67du,21076481.0,FT8
2,SM6YEC,JO67AJ,14075480.0,FT8
3,MI0UFT,IO64IO,14075360.0,FT8
4,KY0R,DM78PF,14075079.0,FT8
...,...,...,...,...
7135,KF0UQR,EM18fe88,50313613.0,FT8
7136,OK1MXD,JN89AX,50313797.0,FT8
7137,PA0RXO,JO20XV,50313624.0,FT8
7138,RA3CQ,KO85sp,50313820.0,FT8


In [18]:
size_bytes = final_report.memory_usage(deep=True).sum()
size_mb = size_bytes / (1024 ** 2)
print(f"DataFrame size: {size_bytes} bytes ({size_mb:.2f} MB)")

DataFrame size: 1084189 bytes (1.03 MB)


In [25]:
with gzip.open('test_points.pkl.gz', 'wb') as f:
    pickle.dump(final_report, f)